In [ ]:
# Load models and split
import joblib

models = joblib.load('../data/processed/baseline_model.pkl')
train_X, val_X, train_y, val_y = joblib.load('../data/processed/train_val_split.pkl')

In [ ]:
# Core metrics table

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score, precision_score
import pandas as pd

results = []
for name, model in models.items():
    probs = model.predict_proba(val_X)[:, 1]
    preds = model.predict(val_X)

    results.append({
        'model': name, 
        'roc_auc': roc_auc_score(val_y, probs),
        'pr_auc': average_precision_score(val_y, probs),
        'f1': f1_score(val_y, preds), 
        'recall': recall_score(val_y, preds),
        'precision': precision_score(val_y, preds)

    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
# ROC and Precision-Recall Curves
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for name, model in models.item():
    probs = model.predict_proba(val_X)[:, 1]
    RocCurveDisplay.from_predictions(val_y, probs, name=name, ax=ax[0])
    PrecisionRecallDisplay.from_predictions(val_y, probs, name=name, ax=ax[1])

plt.show()

In [ ]:
# Cross-Validation Stability Check
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

for name, model in models.items():
    scores = cross_val_score(model, train_X, train_y, cv=cv, scoring='roc_auc')
    print(name, scores.mean(), scores.std())

In [ ]:
# Feature Importance 
rf_importance = pd.Series(models['Random Forest'].feature_importances_, index=train_X.columns).sort_values(ascending=False)
lgbm_importance = pd.Series(models['LightGBM'].feature_importances_, index=train_X.columns).sort_values(ascending=False)

rf_importance.head(10)
lgbm_importance.head(10)

In [ ]:
# Logistic Regression Coefficients
lr_coef = pd.Series(models['Logistic Regression'].coef_[0], index=train_X.columns).sort_values(key=abs, ascending=False)
lr_coef.head(10)

In [ ]:
results_df.to_csv('data/processed/model_validation_results.csv', index=False)